> <p><small><small>Dieses Notebook wird unter der Lizenz und den Bedingungen zur Verfügung gestellt, die in der <a href = "http://www.github.com/google-deepmind/ai-foundations">AI Research Foundations Github README-Datei</a> aufgeführt sind.

<img src="https://storage.googleapis.com/dm-educational/assets/ai_foundations/GDM-Labs-banner-image-C5-white-bg.png">

# Lab: Vollständiges Parameter-Feintuning von Gemma

<a href='https://colab.research.google.com/github/Unisvet/open-weights-project/blob/main/assets/gdm_lab_5_4_full_parameter_fine_tuning_of_gemma_de.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg' alt='Open In Colab'/></a>

15 Minuten

Versuch, ein vollständiges Parameter-Feintuning (Full-Parameter Fine-Tuning) eines Gemma-Modells durchzuführen.

## Übersicht

Im vorherigen Lab haben Sie ein kleines Sprachmodell (SLM) feinjustiert. Wie Sie beobachtet haben, war dieses Modell oft in der Lage, Antworten von angemessener Qualität für Karteikarten zu Themen zu erstellen, die im Africa-Galore-Datensatz enthalten sind. Es schlug jedoch im Allgemeinen fehl, wenn es aufgefordert wurde, Antworten für Karteikarten zu Themen zu generieren, die nicht im Africa-Galore-Datensatz enthalten sind.

In diesem Lab werden Sie versuchen, die Technik des vollständigen Parameter-Feintunings (Full-Parameter Fine-Tuning) auf das vortrainierte Gemma-Modell anzuwenden. Da Gemma auf deutlich mehr Daten trainiert wurde, wird es wahrscheinlich in der Lage sein, nützliche Antworten auf weitaus mehr Fragen zu generieren. Wenn Sie jedoch versuchen, das vollständige Parameter-Feintuning anzuwenden, werden Sie schnell auf eine kritische Einschränkung der Praxis stoßen: Speicherbegrenzungen. Dieses Lab soll Ihnen aus erster Hand demonstrieren, warum das vollständige Parameter-Feintuning bei großen Modellen eine Herausforderung sein kann und warum effizientere Techniken erforderlich sind.

### Was Sie lernen werden
Am Ende dieses Labs werden Sie in der Lage sein:

* Eine Keras-Implementierung eines vortrainierten Gemma-Modells zu laden und damit zu interagieren.
* Die praktischen Speicherbegrenzungen des vollständigen Parameter-Feintunings zu erkennen, wenn es auf Modelle mit Milliarden von Parametern angewendet wird.

### Aufgaben

**In diesem Lab werden Sie**:
* Den Karteikarten-Datensatz (Flashcard Dataset) für die Verwendung mit dem Gemma-Tokenizer und Keras vorbereiten.
* Das vortrainierte Gemma-Modell laden und seine Basisleistung mit einigen Prompts testen.
* Versuchen, ein vollständiges Parameter-Feintuning auf dem Modell durchzuführen.
* Die speicherbedingten Fehler beobachten, die den Erfolg des Trainings verhindern.

⚠️ **Dieses Lab muss auf einer GPU ausgeführt werden. Wählen Sie eine T4-GPU.** Siehe den Abschnitt „Verwendung von Google Colaboratory (Colab)“ unten für Anweisungen zur Durchführung.

## Verwendung von Google Colaboratory (Colab)

Google Colaboratory (auch bekannt als Google Colab) ist eine Plattform, auf der Sie Python-Code in Ihrem Browser ausführen können. Der Code wird in Zellen geschrieben, die auf einem entfernten Server ausgeführt werden.

Um eine Zelle auszuführen, fahren Sie mit der Maus über die Zelle und klicken Sie auf die Schaltfläche „Ausführen“ (Run) auf der linken Seite. Die Ausführen-Schaltfläche ist der Kreis mit dem Dreieck (▶). Alternativ können Sie auch auf eine Zelle klicken und die Tastenkombination Strg+Eingabetaste (bzw. ⌘+Eingabetaste auf einem Mac) verwenden.

Um dies auszuprobieren, führen Sie die folgende Zelle aus. Dies sollte den heutigen Wochentag darunter ausgeben.

In [ ]:
# Zeigt den heutigen Wochentag an
from datetime import datetime
print(f"Heute ist {datetime.today():%A}.")

Beachten Sie, dass die Reihenfolge, in der Sie die Zellen ausführen, eine Rolle spielt. Wenn Sie ein Lab durcharbeiten, stellen Sie sicher, dass Sie alle Zellen der Reihe nach ausführen. Andernfalls funktioniert der Code möglicherweise nicht. Wenn Sie während der Arbeit an einem Lab eine Pause einlegen, trennt Colab möglicherweise die Verbindung; in diesem Fall müssen Sie alle Zellen erneut ausführen, bevor Sie fortfahren. Um dies zu erleichtern, können Sie die Zelle auswählen, an der Sie gerade arbeiten, und dann im Menü oben **Laufzeit → Vorherige ausführen** wählen (oder die Tastenkombination Strg/⌘ + F8 verwenden). Dadurch werden alle Zellen vor der aktuellen erneut ausgeführt.

### Verwendung von Colab mit einer GPU

Befolgen Sie diese Schritte, um die Aktivitäten in diesem Lab auf einer GPU auszuführen:

1. In der oberen Menüleiste auf **Laufzeit** (Runtime) klicken.
2. Wählen Sie **Laufzeittyp ändern** (Change runtime type) aus dem Dropdown-Menü.
3. Wählen Sie im Pop-up-Fenster unter **Hardwarebeschleuniger** (Hardware accelerator) die Option **GPU** (normalerweise als `T4 GPU` aufgeführt).
4. Klicken Sie auf **Speichern** (Save).

Ihre Colab-Sitzung wird nun mit GPU-Zugriff neu gestartet.

Beachten Sie, dass der Zugriff auf GPUs begrenzt ist und Sie dieses Lab zeitweise möglicherweise nicht auf einer GPU ausführen können. Alle Aktivitäten funktionieren trotzdem, laufen jedoch langsamer, und Sie müssen länger warten, bis einige Zellen fertig ausgeführt sind.

## Einrichten eines Kaggle-Kontos

Um dieses Notebook auszuführen, müssen Sie sich bei [Kaggle](https://www.kaggle.com) registrieren – einer Plattform, die Datensätze und Modelle für maschinelles Lernen bereitstellt – und der Nutzungsvereinbarung für das Gemma 3-Modell zustimmen. Dies ist erforderlich, damit Sie die Gewichte des Gemma-Modells für das Feintuning herunterladen können.

### Schritt 1: Erstellen Sie Ihr Kaggle-Konto

* Gehen Sie auf die Kaggle-Website: https://www.kaggle.com

* Klicken Sie auf die Schaltfläche „Register“ in der rechten oberen Ecke.

* Sie können sich mit Ihrem Google-Konto registrieren (empfohlen für eine einfache Colab-Integration) oder eine E-Mail-Adresse und ein Passwort eingeben.

* Folgen Sie den Anweisungen auf dem Bildschirm, um Ihre Registrierung abzuschließen und Ihre E-Mail-Adresse zu bestätigen.

### Schritt 2: Akzeptieren Sie die Gemma 3 Modellvereinbarung

* Stellen Sie sicher, dass Sie in Ihrem neuen Kaggle-Konto angemeldet sind.

* Gehen Sie direkt auf die Gemma 3 Modellseite auf Kaggle: https://www.kaggle.com/models/keras/gemma3/keras/

* Sie sollten eine Schaltfläche „Request Access“ (Zugriff anfordern) sehen.

* Klicken Sie auf die Schaltfläche, lesen Sie die Lizenzvereinbarung durch und klicken Sie auf „Accept“ (Akzeptieren), um Zugriff auf das Modell zu erhalten. Sie müssen dies tun, bevor Sie das Modell über die API herunterladen können.

### Schritt 3: Erstellen Sie Ihren Kaggle-API-Schlüssel

* Klicken Sie auf einer beliebigen Kaggle-Seite auf Ihr Profilbild oder -symbol in der rechten oberen Ecke.

* Wählen Sie „Account“ aus dem Dropdown-Menü.

* Scrollen Sie nach unten zum Bereich „API“.

* Klicken Sie auf die Schaltfläche „Create Legacy API Key“ im Bereich „Legacy API Credentials“.

* Dadurch wird sofort eine Datei namens `kaggle.json` auf Ihren Computer heruntergeladen. Diese Datei enthält Ihren Benutzernamen und Ihren geheimen API-Schlüssel. Bewahren Sie sie sicher auf.

### Schritt 4: Hinterlegen Sie Ihren API-Schlüssel in Colab

* Klicken Sie auf das „Schlüssel“-Symbol 🔑 in der linken Seitenleiste.

* Sie sehen das Panel „Secrets“ (Geheimnisse).

* Öffnen Sie nun die Datei `kaggle.json`, die Sie heruntergeladen haben. Es ist eine einfache Textdatei und sieht etwa so aus:

   ```json
   {"username":"IHR_KAGGLE_BENUTZERNAME","key":"IHR_KAGGLE_API_SCHLÜSSEL"}
   ```
* Erstellen Sie im Colab-Secrets-Panel zwei neue Einträge:

   1. Name: `KAGGLE_USERNAME`

      Wert: Kopieren Sie `IHR_KAGGLE_BENUTZERNAME` aus Ihrer `kaggle.json`-Datei.

   2. Name: `KAGGLE_KEY`

      Wert: Kopieren Sie `IHR_KAGGLE_API_SCHLÜSSEL` aus Ihrer `kaggle.json`-Datei.

* Stellen Sie bei beiden Secrets sicher, dass der Schalter „Notebook access“ (Notebook-Zugriff) aktiviert ist.

## Importe

In diesem Lab verwenden Sie das Keras-Paket zum Laden eines Gemma-Modells sowie das Pandas-Paket zum Laden des Datensatzes.

Führen Sie die folgende Zelle aus, um die erforderlichen Pakete zu importieren.

In [ ]:
%%capture
# Installieren des benutzerdefinierten Pakets für diesen Kurs.
!pip install "git+https://github.com/google-deepmind/ai-foundations.git@main"

import os

# Zum Laden des Kaggle-Benutzernamens und -Schlüssels.
from google.colab import userdata

# Laden des Kaggle-Benutzernamens und -Schlüssels.
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

os.environ["KERAS_BACKEND"] = "jax" # Keras-Backend auf JAX festlegen.

# Deaktivieren der Command-Buffer-Voraballokation, um Speicher freizugeben.
os.environ["XLA_FLAGS"] = "--xla_gpu_enable_command_buffer="
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"

import keras # Für das Training des Modells.
import keras_hub # Zum Laden von Gemma 3.
import pandas as pd # Zum Laden des Datensatzes.
from textwrap import fill # Um Absätze lesbarer zu gestalten.
# Zum Laden der Formatierungsfunktion aus dem Lab
# „Format Text for Turn-Based Dialogue“ (Formatierung für rundenbasierte Dialoge).
from ai_foundations import formatting

keras.utils.set_random_seed(812) # Um das Training reproduzierbar zu machen.

## Motivation

Wie im vorherigen Artikel besprochen, bietet Ihnen das Feintuning von Basismodellen die Möglichkeit, das Wissen zu nutzen, das aus dem Vortraining eines Modells auf riesigen Datenmengen stammt.

Um ein besseres Gefühl dafür zu bekommen, wie sich ein vortrainiertes Modell wie Gemma-1B von dem kleinen Sprachmodell (SLM) unterscheidet, mit dem Sie im vorherigen Lab gearbeitet haben, betrachten Sie die Vergleiche in der folgenden Tabelle:

| | Gemma-1B | SLM |
|----------|----------|----------|
| Anzahl der Parameter | 1.000.000.000 | 523.470 |
| Tokens im Vokabular | 256.000 | 3.086 |
| Tokens im Trainingsset | 2.000.000.000.000 | 92.568 |
| Trainingsdaten | Webdokumente in 140 Sprachen | Africa-Galore-Datensatz |

Wie diese Tabelle zeigt, enthält das Gemma-Modell wesentlich mehr Parameter (~1 Milliarde gegenüber ~500.000) und wurde auf einem Datensatz trainiert, der viel größer und vielfältiger ist als der Africa-Galore-Datensatz. Diese Kombination ermöglicht es dem Modell, mehr Informationen zu lernen und zu speichern als das kleine Sprachmodell. Wenn Sie also Gemma-1B darauf feinjustieren, Antworten für Karteikarten zu generieren, wird es nützliche Antworten für ein viel breiteres Themenspektrum liefern können als das SLM.

## Daten-Vorverarbeitung

Wie bei Ihrem SLM müssen Sie sicherstellen, dass Ihre Feintuning-Daten den Tokenizer und die speziellen Begrenzungstoken verwenden, die während des Vortrainings verwendet wurden. In diesem Kurs verwenden Sie das Format, das instruktions-abgestimmte Gemma-Modelle verwenden (eingeführt im Lab „Formatting“ zur rundenbasierten Formatierung) [1]. Es verwendet spezielle Token, um den Beginn und das Ende einer Runde zu markieren, und gibt die Rolle an (`user` oder `model`).

Ein einzelnes Trainingsbeispiel ist wie folgt strukturiert:

------
> `<start_of_turn>`user
>
> What is Jollof rice?`<end_of_turn>`
>
> `<start_of_turn>`model
>
> Category: Food
>
> Jollof rice is a popular and iconic one-pot rice dish that is a staple in many West African countries.`<end_of_turn>`
------

Führen Sie die folgende Zelle aus, um den Africa-Galore-QA-Datensatz zu laden und jedes Beispiel so zu formatieren, dass es diese Struktur aufweist.

In [ ]:
# Laden des Frage-Antwort-Datensatzes.
africa_galore_qa = pd.read_json(
    "https://storage.googleapis.com/dm-educational/assets/ai_foundations/africa_galore_qa_v2.json"
)

questions = []  # Liste der formatierten Fragen.
answers = []  # Liste der formatierten Antworten.

for idx, row in africa_galore_qa.iterrows():
    # Führen Sie die format_qa-Funktion aus dem vorherigen Lab aus, um die Frage
    # und die Antwort zu formatieren.
    question, answer = formatting.format_qa(row)
    questions.append(question)
    answers.append(answer)

# Zeigt das erste Set von Inputs und Outputs.
print(questions[0])
print(fill(answers[0], replace_whitespace=False))

Erinnern Sie sich daran, dass Sie im vorherigen Lab zusätzliche Schritte zur Vorbereitung der Daten durchführen mussten. Sie mussten außerdem jede Frage und jede Antwort tokenisieren und manuell einige Token im Ziel durch das spezielle `<PAD>`-Token ersetzen, um es von der Verlustberechnung (Loss) auszuschließen.

Da das Feintuning eines Modells eine sehr häufige Aufgabe ist, enthalten Implementierungen von Modellen wie Gemma oft bereits Funktionen, die mehrere Schritte der Datenvorbereitung im Rahmen eines Preprozessors übernehmen.

Im Fall der Keras-Implementierung des Gemma-Modells müssen Sie lediglich den Prompt angeben und das, was das Modell generieren soll. Sie erstellt dann automatisch tokenisierte Beispiele, die zum Feintuning eines Gemma-Sprachmodells verwendet werden können. Im Hintergrund werden Prompts und Antworten verkettet und die Token im Prompt von der Verlustberechnung ausgeschlossen. Dies erspart Ihnen viele gängige Schritte. Beachten Sie jedoch, dass `<start_of_turn>`- oder `<end_of_turn>`-Token nicht automatisch hinzugefügt werden, sodass Sie diese Token dem Modell weiterhin bereitstellen müssen, wie Sie es in der vorherigen Zelle getan haben.

### Vorbereiten der Daten für das Keras-Modell

Die Keras-Implementierung von Gemma erwartet, dass die Feintuning-Daten als Python-Dictionary mit zwei spezifischen Schlüsseln strukturiert sind: `prompts` für die Prompts (Fragen im Fall des Karteikarten-Generators) und `responses` für die Ausgaben, die das Modell generieren soll (Antworten im Fall des Karteikarten-Generators).

Führen Sie die folgende Zelle aus, um dieses Dictionary zu initialisieren.

In [ ]:
data = {
    "prompts": questions,
    "responses": answers
}

## Laden des Gemma-Modells mit Keras

Die folgende Zelle lädt das vortrainierte [Gemma-1B-Modell](https://ai.google.dev/gemma/docs/core/prompt-structure) mit Keras [2]. Wie Ihr SLM wurde dieses Modell für die Vorhersage des nächsten Wortes (Next-Word Prediction) trainiert und ist nicht für die Beantwortung von Fragen optimiert. Nach dem Laden können Sie eine Zusammenfassung mit der Methode `summary` ausgeben, um die Architektur zu überprüfen.

------
> **💭 Reflexion: Speicheranforderungen**
>
> Berücksichtigen Sie vor dem Ausführen des Codes die folgenden Fragen:
>
> * Wie viele Parameter erwarten Sie für das Modell?
> * Wenn Sie ein Colab-Notebook mit einer T4-GPU verwenden, verfügt diese über ca. 15 GB Speicher. Werden Sie in der Lage sein, das gesamte Modell in den Speicher Ihrer GPU zu laden?
> * Würde ein viel größeres Modell wie Gemma-27B auf Ihre GPU passen?
>
------

Führen Sie die folgende Zelle aus, um das Modell zu laden und den Preprozessor zu initialisieren.

In [ ]:
# Laden des Gemma3-1B Keras-Modells.
model = keras_hub.models.Gemma3CausalLM.from_preset("gemma3_1b")
model.summary()

# Maximale Sequenzlänge des Modells für Padding und Batching festlegen.
model.preprocessor.sequence_length = 400

#### Gemmas Speicherbedarf verstehen

Das Gemma 1B-Modell hat insgesamt etwa 1 Kunde/Milliarde Parameter (ca. 700 Mio. für die Haupt-Transformer-Blöcke und 300 Mio. für die Token-Embeddings, die von Input und Output gemeinsam genutzt werden). Ein Modell dieser Größe benötigt typischerweise etwa 4 GB Speicher, sodass es problemlos auf eine T4-GPU mit 15 GB Speicher passt. Ein 27B-Modell hingegen – also ein Modell mit rund 27 Milliarden Parametern – würde etwa 100 GB Speicher benötigen (27 $\times$ ~3.7 GB), was die Kapazität einer einzelnen T4-GPU um ein Vielfaches übersteigt.

## Promptern des Basismodells

Bevor Sie das Modell feinjustieren, beobachten Sie, wie sich das vortrainierte Basismodell verhält, wenn Sie ihm eine Frage stellen.

In [ ]:
prompt = "What is Jollof rice?"
response = model.generate(prompt, max_length=200)
print(fill(response, replace_whitespace=False))

### Was haben Sie beobachtet?

Bei einem Prompt wie „What is Jollof rice?“ haben Sie wahrscheinlich festgestellt, dass die Antwort des Modells Wiederholungen enthält, nicht prägnant ist und vom Thema abweichen kann. Darüber hinaus gibt es die Antwort nicht im gewünschten Karteikarten-Format aus.

Diese Mängel sind zu erwarten. Denken Sie daran, dass dieses Modell auf großen Textmengen trainiert wurde, um das nächste Token vorherzusagen. Im Gegensatz zu instruktions-abgestimmten Modellen wurde es jedoch nicht dafür optimiert, auf Benutzerfragen zu antworten. Darüber hinaus enthält sein Vortraining keine Informationen über das Format des Karteikarten-Generators.

Durch das Feintuning eines Modells können Sie beide Probleme beheben. Das Modell lernt, prägnantere Antworten auf Fragen zu geben und das Karteikarten-Format einzuhalten, da beide Eigenschaften in den Trainingsbeispielen des Africa-Galore-QA-Datensatzes demonstriert werden.

## Versuch eines vollständigen Parameter-Feintunings

Das Feintuning erfordert erheblich mehr Speicher als die Nutzung eines Modells für Inferenz. Für jeden Parameter muss der Trainingsprozess nicht nur den aktuellen Wert des Parameters selbst speichern, sondern auch den Gradienten und andere vom Optimizer verwendete Variablen. Aus diesem Grund benötigt das Training eines Modells ein Vielfaches an Speicherplatz im Vergleich zur reinen Texterzeugung.

Führen Sie für diesen Schritt den folgenden Code aus, um das Gemma-1B-Modell mit demselben vollständigen Parameter-Ansatz (Full-Parameter Fine-Tuning) feinjustieren zu lassen, den Sie auch für das Feintuning Ihres SLMs verwendet haben. Der folgende Code aktualisiert alle Parameter von Gemma.

Beachten Sie: Wenn beim Ausführen des Feintunings ein Fehler auftritt, lesen Sie bitte weiter für Erklärungen und Lösungen.

In [ ]:
model.fit(data, epochs=1, batch_size=32, verbose=1)

### Was haben Sie beobachtet?

Beim Ausführen der obigen Zelle sollten Sie einen „Out of Memory“ (OOM)-Fehler beobachten. Dies geschah, weil der Optimizer versuchte, mehr Speicher zuzuweisen, als auf einer T4-GPU verfügbar ist, während er versuchte, alle 1 Milliarde Parameter des Gemma-1B-Modells feinjustieren zu lassen. Wenn dies passiert, stürzt die gesamte Colab-Sitzung ab und Sie müssen den gesamten Code erneut ausführen.

Dies ist ein bewusster Schritt, der verdeutlicht, dass Sie zwar Modelle mit mehr als 1B Parametern laden und Inferenz auf einer T4-GPU mit 15 GB durchführen können (wie Sie es zuvor in diesem Lab getan haben), dies jedoch nicht bedeutet, dass Sie auch alle Parameter eines solchen Modells feinjustieren können. Wie Sie in zukünftigen Kursen genauer lernen werden, erfordert jede Form des Modelltrainings, einschließlich des Feintunings, erheblich mehr Speicher, da Sie auch Gradienten und Informationen für den Optimizer im Speicher der GPU hinterlegen müssen.

## Zusammenfassung

In diesem Lab haben Sie mit dem vortrainierten Gemma-1B-Modell experimentiert. Sie haben versucht, es mittels vollständigem Parameter-Feintuning an eine neue Aufgabe anzupassen, und festgestellt, dass diese Methode für große Modelle aufgrund der enormen Speicheranforderungen oft unpraktisch ist.

Um dies zu lösen, benötigen Sie einen raffinierteren Ansatz. Die kommenden Labs werden Sie in das Konzept des parameter-effizienten Feintunings (PEFT) und eine Technik namens LoRA einführen, die es Ihnen ermöglicht, große Modelle wie Gemma mit begrenztem GPU-Speicher feinjustieren zu lassen.

## Referenzen

[1] Google AI - Gemma Prompt Structure
Google AI. 2025. Gemma formatting and system instructions. Abgerufen von https://ai.google.dev/gemma/docs/core/prompt-structure

[2] Keras Hub - Gemma3CausalLM
Keras. 2025. Gemma3CausalLM model. Abgerufen von https://keras.io/keras_hub/api/models/gemma3/gemma3_causal_lm/